<a href="https://colab.research.google.com/github/DanielRegaladoUMiami/mayavoice-llm/blob/main/notebooks/02_finetune_v2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🌽 MayaVoice v2 — Fine-tuning Llama 3.1-8B con Corpus Aumentado
## Sprint 2: 14 idiomas mayas, datos bíblicos + augmentación por sinónimos

**Cambios vs v1:**
- **14 idiomas** (se agregó Kaqchikel vía corpus bíblico paralelo)
- **120,656 pares paralelos** → 241,312 ejemplos de entrenamiento (3.7x más que v1)
- **Augmentación por sinónimos** (WordNet español) para multiplicar datos
- **Corpus bíblico** Kaqchikel + Mam (Nuevo Testamento completo)

**Datos v2:** `train.jsonl` (193K), `val.jsonl` (24K), `test_with_meta.jsonl` (24K)

In [2]:
# ============================================================
# 1. INSTALACIÓN (solo ejecutar una vez por sesión de Colab)
# ============================================================
%%capture
!pip install unsloth
!wget -qO- https://raw.githubusercontent.com/unslothai/unsloth/main/unsloth/_auto_install.py | python -
!pip install sacrebleu

In [3]:
# ============================================================
# 2. CONFIGURACIÓN v2
# ============================================================
import json
import random
import torch
from pathlib import Path

CONFIG = {
    "base_model": "unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit",
    "max_seq_length": 2048,
    "lora_r": 32,           # v2: aumentado de 16 → 32 (más capacidad para más datos)
    "lora_alpha": 64,       # v2: 2x lora_r
    "lora_dropout": 0.05,
    "per_device_train_batch_size": 4,
    "gradient_accumulation_steps": 4,
    "learning_rate": 1e-4,   # v2: reducido de 2e-4 → 1e-4 (más datos = lr más bajo)
    "warmup_ratio": 0.05,   # v2: reducido (dataset más grande calienta más rápido)
    "num_train_epochs": 2,   # v2: reducido de 3 → 2 (3.7x más datos compensa)
    "seed": 42,
    "output_dir": "./mayavoice-lora-v2",
    "hub_model_id": "DanielRegaladoCardoso/mayavoice-llama3.1-8b-lora-v2",
    "version": "v2",
}

random.seed(CONFIG["seed"])
torch.manual_seed(CONFIG["seed"])
print(f"GPU: {torch.cuda.get_device_name(0)}")
vram = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"VRAM: {vram:.1f} GB")
print(f"\n🏷️  Versión: {CONFIG['version']}")
print(f"   LoRA r={CONFIG['lora_r']}, alpha={CONFIG['lora_alpha']}")
print(f"   LR={CONFIG['learning_rate']}, epochs={CONFIG['num_train_epochs']}")

GPU: NVIDIA A100-SXM4-40GB
VRAM: 42.4 GB

🏷️  Versión: v2
   LoRA r=32, alpha=64
   LR=0.0001, epochs=2


In [4]:
# ============================================================
# 3. CARGAR MODELO BASE CON UNSLOTH
# ============================================================
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=CONFIG["base_model"],
    max_seq_length=CONFIG["max_seq_length"],
    dtype=None,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=CONFIG["lora_r"],
    lora_alpha=CONFIG["lora_alpha"],
    lora_dropout=CONFIG["lora_dropout"],
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=CONFIG["seed"],
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Parámetros entrenables: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.3.4: Fast Llama patching. Transformers: 5.2.0.
   \\   /|    NVIDIA A100-SXM4-40GB. Num GPUs = 1. Max memory: 39.494 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/5.70G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

Unsloth: Will load unsloth/meta-llama-3.1-8b-instruct-bnb-4bit as a legacy tokenizer.
Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.05.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.
Unsloth 2026.3.4 patched 32 layers with 0 QKV layers, 0 O layers and 0 MLP layers.


Parámetros entrenables: 83,886,080 / 4,624,486,400 (1.81%)


In [6]:
# ============================================================
# 4. CARGAR DATOS v2 DESDE GOOGLE DRIVE
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

import os

# v2: Buscar splits-v2 primero, luego caer a la carpeta base
POSSIBLE_PATHS = [
    "/content/drive/MyDrive/mayavoice-data/splits-v2",
    "/content/drive/MyDrive/mayavoice-data",
]

DATA_DIR = None
for path in POSSIBLE_PATHS:
    if os.path.exists(path) and os.path.isfile(f"{path}/train.jsonl"):
        DATA_DIR = path
        break

if DATA_DIR is None:
    import subprocess
    result = subprocess.run(["find", "/content/drive", "-name", "train.jsonl", "-maxdepth", 6],
                          capture_output=True, text=True, timeout=30)
    if result.stdout.strip():
        DATA_DIR = os.path.dirname(result.stdout.strip().split("\n")[0])

if DATA_DIR:
    print(f"✅ Datos v2 encontrados en: {DATA_DIR}")
    for f in sorted(os.listdir(DATA_DIR)):
        if f.endswith(('.jsonl', '.json')):
            size = os.path.getsize(f"{DATA_DIR}/{f}") / (1024*1024)
            print(f"   {f} ({size:.1f} MB)")
else:
    raise FileNotFoundError("❌ No se encontraron los datos v2. Sube train.jsonl, val.jsonl, test_with_meta.jsonl a Drive/mayavoice-data/splits-v2/")

def load_jsonl(path):
    data = []
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            data.append(json.loads(line))
    return data

train_data = load_jsonl(f"{DATA_DIR}/train.jsonl")
val_data = load_jsonl(f"{DATA_DIR}/val.jsonl")

print(f"\nTrain: {len(train_data):,} ejemplos")
print(f"Val:   {len(val_data):,} ejemplos")
print(f"\nEjemplo:")
print(json.dumps(train_data[0], indent=2, ensure_ascii=False))

Mounted at /content/drive
✅ Datos v2 encontrados en: /content/drive/MyDrive/mayavoice-data/splits-v2
   test_with_meta.jsonl (6.9 MB)
   train.jsonl (46.2 MB)
   val.jsonl (5.8 MB)

Train: 193,038 ejemplos
Val:   24,116 ejemplos

Ejemplo:
{
  "instruction": "Convierte este texto del Poqomchi' al español.",
  "input": "Nujinaq chi haʼ i ikom",
  "output": "La tinaja está llena de lluvia"
}


In [7]:
# ============================================================
# 5. FORMATEAR DATOS PARA LLAMA 3.1 INSTRUCT
# ============================================================
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(
    tokenizer,
    chat_template="llama-3.1",
)

def format_example(example):
    messages = [
        {"role": "system", "content": "Eres MayaVoice, un asistente especializado en idiomas mayas de Guatemala. Traduces con precisión y respetas la riqueza lingüística de cada idioma."},
        {"role": "user", "content": f"{example['instruction']}\n\n{example['input']}"},
        {"role": "assistant", "content": example['output']},
    ]
    return {"messages": messages}

train_formatted = [format_example(ex) for ex in train_data]
val_formatted = [format_example(ex) for ex in val_data]

print("Ejemplo formateado:")
for msg in train_formatted[0]["messages"]:
    print(f"  [{msg['role']}]: {msg['content'][:80]}...")

Ejemplo formateado:
  [system]: Eres MayaVoice, un asistente especializado en idiomas mayas de Guatemala. Traduc...
  [user]: Convierte este texto del Poqomchi' al español.

Nujinaq chi haʼ i ikom...
  [assistant]: La tinaja está llena de lluvia...


In [8]:
# ============================================================
# 6. TOKENIZAR DATASET
# ============================================================
from unsloth.chat_templates import standardize_sharegpt, train_on_responses_only
from datasets import Dataset

def apply_chat_template(examples):
    texts = []
    for msgs in examples["messages"]:
        text = tokenizer.apply_chat_template(
            msgs, tokenize=False, add_generation_prompt=False
        )
        texts.append(text)
    return {"text": texts}

train_dataset = Dataset.from_list(train_formatted)
val_dataset = Dataset.from_list(val_formatted)

train_dataset = train_dataset.map(apply_chat_template, batched=True)
val_dataset = val_dataset.map(apply_chat_template, batched=True)

lengths = [len(tokenizer.encode(t)) for t in train_dataset["text"][:1000]]
print(f"Longitud promedio: {sum(lengths)/len(lengths):.0f} tokens")
print(f"Max: {max(lengths)}, Min: {min(lengths)}")
print(f"% > 2048 tokens: {100*sum(1 for l in lengths if l > 2048)/len(lengths):.1f}%")

Map:   0%|          | 0/193038 [00:00<?, ? examples/s]

Map:   0%|          | 0/24116 [00:00<?, ? examples/s]

Longitud promedio: 156 tokens
Max: 423, Min: 99
% > 2048 tokens: 0.0%


In [10]:
# ============================================================
# 7. ENTRENAR CON SFTTrainer
# ============================================================
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    dataset_text_field="text",
    max_seq_length=CONFIG["max_seq_length"],
    dataset_num_proc=2,
    packing=False,
    args=TrainingArguments(
        output_dir=CONFIG["output_dir"],
        per_device_train_batch_size=CONFIG["per_device_train_batch_size"],
        gradient_accumulation_steps=CONFIG["gradient_accumulation_steps"],
        learning_rate=CONFIG["learning_rate"],
        warmup_ratio=CONFIG["warmup_ratio"],
        num_train_epochs=CONFIG["num_train_epochs"],
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=50,
        eval_strategy="no",
        save_strategy="steps",
        save_steps=2000,
        optim="adamw_8bit",
        lr_scheduler_type="cosine",
        seed=CONFIG["seed"],
        report_to="none",
    ),
)

trainer = train_on_responses_only(
    trainer,
    instruction_part="<|start_header_id|>user<|end_header_id|>\n\n",
    response_part="<|start_header_id|>assistant<|end_header_id|>\n\n",
)

print("🚀 Iniciando entrenamiento v2...")
print(f"   {len(train_dataset):,} train / {len(val_dataset):,} val")
print(f"   Effective batch = {CONFIG['per_device_train_batch_size'] * CONFIG['gradient_accumulation_steps']}")
stats = trainer.train()
print(f"\n✅ Entrenamiento v2 completado!")
print(f"   Tiempo total: {stats.metrics['train_runtime']/60:.1f} minutos")
print(f"   Loss final: {stats.metrics['train_loss']:.4f}")

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Unsloth: Tokenizing ["text"] (num_proc=16):   0%|          | 0/193038 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=16):   0%|          | 0/24116 [00:00<?, ? examples/s]

Map (num_proc=16):   0%|          | 0/193038 [00:00<?, ? examples/s]

Filter (num_proc=16):   0%|          | 0/193038 [00:00<?, ? examples/s]

...

Filter (num_proc=16):   0%|          | 0/24116 [00:00<?, ? examples/s]

🚀 Iniciando entrenamiento v2...
   193,038 train / 24,116 val
   Effective batch = 16


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 193,038 | Num Epochs = 2 | Total steps = 24,130
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 4 x 1) = 16
 "-____-"     Trainable parameters = 83,886,080 of 8,114,147,328 (1.03% trained)


Step,Training Loss
50,4.314803
100,3.735928
150,3.300356
200,3.089897
250,2.944728
300,2.717794
350,2.651059
400,2.627227
450,2.498134
500,2.515734



✅ Entrenamiento v2 completado!
   Tiempo total: 708.1 minutos
   Loss final: 0.8874


In [11]:
# ============================================================
# 8. EVALUACIÓN RÁPIDA — TRADUCCIONES DE PRUEBA
# ============================================================
FastLanguageModel.for_inference(model)

test_cases = [
    ("Traduce del español al K'iche'.", "Buenos días, ¿cómo estás?"),
    ("Traduce del español al Mam.", "La tierra es sagrada para nosotros."),
    ("Traduce del español al Q'eqchi'.", "Los niños están jugando en el campo."),
    ("Traduce del K'iche' al español.", "Saqarik, la utz awach?"),
    ("Traduce del español al Tz'utujil.", "El maíz ya está listo para cosechar."),
    ("Traduce del español al Kaqchikel.", "Gracias por tu ayuda."),
    ("Traduce del Kaqchikel al español.", "Matyox chawe."),
]

for instruction, input_text in test_cases:
    messages = [
        {"role": "system", "content": "Eres MayaVoice, un asistente especializado en idiomas mayas de Guatemala."},
        {"role": "user", "content": f"{instruction}\n\n{input_text}"},
    ]

    input_ids = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True,
        return_tensors="pt"
    )
    if hasattr(input_ids, "input_ids"):
        input_ids = input_ids["input_ids"]
    input_ids = input_ids.to("cuda")

    outputs = model.generate(
        input_ids=input_ids, max_new_tokens=128, temperature=0.3,
        use_cache=True, do_sample=True,
    )

    response = tokenizer.decode(outputs[0][input_ids.shape[-1]:], skip_special_tokens=True)
    print(f"📝 {instruction}")
    print(f"   Input:  {input_text}")
    print(f"   Output: {response}")
    print()

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
--- Logging error ---
Traceback (most recent call last):
  File "/usr/lib/python3.12/logging/__init__.py", line 1160, in emit
    msg = self.format(record)
          ^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/logging/__init__.py", line 999, in format
    return fmt.format(record)
           ^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/logging/__init__.py", line 703, in format
    record.message = record.getMessage()
                     ^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/logging/__init__.py", line 392, in getMessage
    msg = msg % self.args
          ~~~~^~~~~~~~~~~
TypeError: not all arguments converted during string formatting
Call stack:
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_c

📝 Traduce del español al K'iche'.
   Input:  Buenos días, ¿cómo estás?
   Output: ¿Jampaʼ katchʼaw? Uj kʼo wi jun utzilaj bʼenaq

📝 Traduce del español al Mam.
   Input:  La tierra es sagrada para nosotros.
   Output: Xjan txʼotxʼ atza tuʼn qetz

📝 Traduce del español al Q'eqchi'.
   Input:  Los niños están jugando en el campo.
   Output: Yookebʼ chi bʼatzʼunk li kokʼal saʼ li chʼochʼ

📝 Traduce del K'iche' al español.
   Input:  Saqarik, la utz awach?
   Output: ¿Estás bien, blanco?

📝 Traduce del español al Tz'utujil.
   Input:  El maíz ya está listo para cosechar.
   Output: Ja ixiim chʼajoon chik chi bʼanoj rikʼin

📝 Traduce del español al Kaqchikel.
   Input:  Gracias por tu ayuda.
   Output: Matiox chawe ruma ri asilo.

📝 Traduce del Kaqchikel al español.
   Input:  Matyox chawe.
   Output: ¡Estás bien razonado!



In [12]:
# ============================================================
# 9. MÉTRICAS v2 — BLEU y chrF
# ============================================================
import sacrebleu
from collections import defaultdict

test_meta = load_jsonl(f"{DATA_DIR}/test_with_meta.jsonl")
print(f"Test set: {len(test_meta):,} ejemplos")

SAMPLE_SIZE = 300  # v2: más muestras para mejor estimación
random.seed(42)
test_sample = random.sample(test_meta, min(SAMPLE_SIZE, len(test_meta)))

FastLanguageModel.for_inference(model)

results_by_lang = defaultdict(lambda: {"refs": [], "hyps": [], "direction": []})

for i, example in enumerate(test_sample):
    messages = [
        {"role": "system", "content": "Eres MayaVoice, un asistente especializado en idiomas mayas de Guatemala."},
        {"role": "user", "content": f"{example['instruction']}\n\n{example['input']}"},
    ]

    input_ids = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True,
        return_tensors="pt"
    )
    if hasattr(input_ids, "input_ids"):
        input_ids = input_ids["input_ids"]
    input_ids = input_ids.to("cuda")

    outputs = model.generate(
        input_ids=input_ids, max_new_tokens=128, temperature=0.1,
        use_cache=True, do_sample=False,
    )

    hyp = tokenizer.decode(outputs[0][input_ids.shape[-1]:], skip_special_tokens=True).strip()
    ref = example["output"].strip()
    lang = example.get("_lang", "unknown")
    direction = example.get("_direction", "unknown")

    results_by_lang[lang]["refs"].append(ref)
    results_by_lang[lang]["hyps"].append(hyp)
    results_by_lang[lang]["direction"].append(direction)

    if (i + 1) % 50 == 0:
        print(f"  Procesados {i+1}/{len(test_sample)}...")

print(f"\n{'='*70}")
print(f"{'MÉTRICAS v2 — MayaVoice Sprint 2':^70}")
print(f"{'='*70}")
print(f"{'Idioma':<15} {'N':<6} {'BLEU':<10} {'chrF':<10}")
print("-" * 41)

all_refs, all_hyps = [], []
for lang in sorted(results_by_lang.keys()):
    data = results_by_lang[lang]
    refs = data["refs"]
    hyps = data["hyps"]
    all_refs.extend(refs)
    all_hyps.extend(hyps)

    bleu = sacrebleu.corpus_bleu(hyps, [refs])
    chrf = sacrebleu.corpus_chrf(hyps, [refs])
    print(f"{lang:<15} {len(refs):<6} {bleu.score:<10.2f} {chrf.score:<10.2f}")

bleu_all = sacrebleu.corpus_bleu(all_hyps, [all_refs])
chrf_all = sacrebleu.corpus_chrf(all_hyps, [all_refs])
print("-" * 41)
print(f"{'TOTAL':<15} {len(all_refs):<6} {bleu_all.score:<10.2f} {chrf_all.score:<10.2f}")

metrics = {
    "model": CONFIG["base_model"],
    "version": "v2",
    "overall_bleu": bleu_all.score,
    "overall_chrf": chrf_all.score,
    "sample_size": len(test_sample),
    "per_language": {}
}
for lang in sorted(results_by_lang.keys()):
    data = results_by_lang[lang]
    bleu = sacrebleu.corpus_bleu(data["hyps"], [data["refs"]])
    chrf = sacrebleu.corpus_chrf(data["hyps"], [data["refs"]])
    metrics["per_language"][lang] = {
        "bleu": bleu.score, "chrf": chrf.score, "n": len(data["refs"])
    }

metrics_path = f"{DATA_DIR}/baseline_metrics_v2.json"
with open(metrics_path, "w") as f:
    json.dump(metrics, f, indent=2)
print(f"\n✅ Métricas v2 guardadas: {metrics_path}")

Test set: 24,158 ejemplos
  Procesados 50/300...
  Procesados 100/300...
  Procesados 150/300...
  Procesados 200/300...
  Procesados 250/300...
  Procesados 300/300...

                   MÉTRICAS v2 — MayaVoice Sprint 2                   
Idioma          N      BLEU       chrF      
-----------------------------------------
achi            4      8.30       44.68     
awakateko       12     15.46      34.46     
chuj            11     31.32      54.78     
itza            7      4.02       17.21     
kaqchikel       57     41.34      56.29     
kiche           18     21.32      38.57     
mam             70     47.07      61.30     
poqomam         24     11.21      48.97     
poqomchi        7      15.15      31.88     
qanjobal        23     23.38      43.92     
qeqchi          22     27.25      52.03     
sipakapense     7      15.75      48.87     
tektiteko       20     15.76      43.78     
tzutujil        18     2.86       28.81     
-----------------------------------------


In [13]:
# ============================================================
# 10. GUARDAR MODELO v2
# ============================================================
import os
from huggingface_hub import login

HF_TOKEN = ...

# --- Guardar en Drive ---
DRIVE_MODEL_DIR = "/content/drive/MyDrive/mayavoice-data/mayavoice-lora-v2"
model.save_pretrained(DRIVE_MODEL_DIR)
tokenizer.save_pretrained(DRIVE_MODEL_DIR)
print(f"✅ LoRA v2 guardado en Drive: {DRIVE_MODEL_DIR}/")

# --- Subir a HuggingFace Hub ---
if HF_TOKEN:
    login(token=HF_TOKEN)
    model.push_to_hub(CONFIG["hub_model_id"], token=HF_TOKEN)
    tokenizer.push_to_hub(CONFIG["hub_model_id"], token=HF_TOKEN)
    print(f"✅ Modelo v2 subido a https://huggingface.co/{CONFIG['hub_model_id']}")
else:
    print("⚠️  No se encontró HF_TOKEN.")
    print("   Opción 1: Crea .env en Drive/mayavoice-data/ con HF_TOKEN=tu_token")
    print("   Opción 2: Ejecuta: import os; os.environ['HF_TOKEN'] = 'tu_token'")

✅ LoRA v2 guardado en Drive: /content/drive/MyDrive/mayavoice-data/mayavoice-lora-v2/


README.md:   0%|          | 0.00/589 [00:00<?, ?B/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:   0%|          |  554kB /  336MB            

Saved model to https://huggingface.co/DanielRegaladoCardoso/mayavoice-llama3.1-8b-lora-v2


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...mp17m92fel/tokenizer.json:  92%|#########2| 15.9MB / 17.2MB            

✅ Modelo v2 subido a https://huggingface.co/DanielRegaladoCardoso/mayavoice-llama3.1-8b-lora-v2


## 📊 Resumen v2 vs v1

| Métrica | v1 | v2 |
|---------|----|----|
| Idiomas | 13 | **14** (+Kaqchikel) |
| Pares paralelos | 32,328 | **120,656** |
| Ejemplos training | 64,656 | **241,312** |
| Train split | 51,714 | **193,038** |
| LoRA rank | 16 | **32** |
| Learning rate | 2e-4 | **1e-4** |
| Epochs | 3 | **2** |

### Fuentes de datos v2:
- MayanV corpus (12 idiomas, CC0)
- Bible corpus — Kaqchikel NT (7,851 versos) + Mam NT (7,789 versos)
- Augmentación por sinónimos (WordNet español, ~1.5x multiplicador)

### Próximos pasos (Sprint 3):
- [ ] RAG pipeline con diccionarios (5,224 entradas + audio)
- [ ] Prompt templates optimizados
- [ ] Test set manual con hablantes nativos (UVG)
- [ ] Comparar v1 vs v2 métricas BLEU/chrF